# Cats vs. Dogs Classification with ResNet18 (Transfer Learning)

Binary image classifier built with PyTorch, using a pretrained ResNet18 backbone and a custom classification head.

**Pipeline:** download → data cleaning → augmentation → train/dev/test split → training → final evaluation on unseen test data.

> Designed to run on Google Colab with a GPU (Runtime → Change runtime type → GPU).

In [ ]:
import os
import time
import torch
import torch.nn as nn
from PIL import Image
from torchvision import datasets, transforms
import torchvision.models as models
from torch.utils.data import DataLoader, random_split

## 1. Download the dataset
Microsoft PetImages (Kaggle Cats and Dogs), ~25,000 images.

In [ ]:
if not os.path.exists('./dataset/PetImages'):
    print("Downloading dataset from Microsoft...")
    !wget -q https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
    print("Unzipping images...")
    !unzip -q kagglecatsanddogs_5340.zip -d ./dataset
    !rm kagglecatsanddogs_5340.zip
    print("Dataset downloaded and extracted successfully.")

## 2. Data cleaning
The raw dataset contains corrupted and zero-byte images that crash training with `UnidentifiedImageError`.
This step scans every file, fully loads it into memory, and removes anything that cannot be decoded or is not an image.

In [ ]:
print("Scanning for corrupted or unidentifiable images...")
removed_count = 0

for pet in ['Cat', 'Dog']:
    folder = f'./dataset/PetImages/{pet}'
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)

        # Remove non-image files (e.g. hidden system files)
        if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                os.remove(file_path)
                removed_count += 1
            except OSError:
                pass
            continue

        # Fully load and convert to RGB to catch corrupted files
        try:
            with Image.open(file_path) as img:
                img.load()
                img.convert('RGB')
        except Exception as e:
            print(f"  Removing corrupted file: {file_path} ({type(e).__name__})")
            try:
                os.remove(file_path)
                removed_count += 1
            except OSError as delete_error:
                print(f"  Failed to delete {file_path}: {delete_error}")

print(f"Scan finished. Removed {removed_count} problematic files.")

## 3. Data augmentation
In the first version the model learned the clean training images too quickly and overfit (training accuracy kept rising while validation accuracy plateaued).
To force the model to learn more robust features, the training set uses aggressive augmentation:

- **TrivialAugmentWide** – random geometric and color transforms (rotation, shear, translation, brightness, contrast, saturation, hue)
- **Random horizontal flip**
- **Gaussian blur** – removes fine detail
- **Random erasing (Cutout)** – hides random patches, forcing the model to rely on different parts of the animal

Validation and test images are only resized and normalized.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.TrivialAugmentWide(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.0))
    ], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.25), ratio=(0.3, 3.3), value='random')
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 4. Train / Dev / Test split (98% / 1% / 1%)
A fixed random seed makes the split reproducible.

In [ ]:
full_dataset_train = datasets.ImageFolder(root='./dataset/PetImages', transform=train_transform)
full_dataset_val = datasets.ImageFolder(root='./dataset/PetImages', transform=val_test_transform)

total_size = len(full_dataset_train)
train_size = int(0.98 * total_size)
val_size = int(0.01 * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_idx, val_idx, test_idx = random_split(range(total_size), [train_size, val_size, test_size], generator=generator)

train_dataset = torch.utils.data.Subset(full_dataset_train, train_idx)
val_dataset = torch.utils.data.Subset(full_dataset_val, val_idx)
test_dataset = torch.utils.data.Subset(full_dataset_val, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Dataset split:")
print(f"  Train: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"  Dev:   {len(val_dataset)} images ({len(val_loader)} batches)")
print(f"  Test:  {len(test_dataset)} images ({len(test_loader)} batches)")

## 5. Model: pretrained ResNet18 + custom head
The pretrained backbone is frozen, and only a new classification head is trained:
`Linear(512 → 256) → ReLU → Dropout(0.5) → Linear(256 → 2)`

Optimizer: **AdamW** (lr = 0.001, weight decay = 0.01). Loss: **Cross-Entropy** (softmax is applied inside the loss).

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze the pretrained backbone
for param in model.parameters():
    param.requires_grad = False

# Replace the classification head
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(256, 2)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=0.001, betas=(0.9, 0.999), weight_decay=0.01)

## 6. Training loop with Dev evaluation

In [ ]:
epochs = 5

for epoch in range(epochs):
    start_time = time.time()

    # --- Train ---
    model.train()
    running_loss, running_corrects, total_train = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total_train += inputs.size(0)

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = (running_corrects.double() / total_train) * 100

    # --- Dev ---
    model.eval()
    val_loss, val_corrects, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            total_val += inputs.size(0)

    epoch_dev_loss = val_loss / total_val
    epoch_dev_acc = (val_corrects.double() / total_val) * 100

    mins, secs = divmod(time.time() - start_time, 60)
    print(f"Epoch {epoch+1}/{epochs} | {int(mins)}m {int(secs)}s")
    print(f"  Train  Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc:.2f}%")
    print(f"  Dev    Loss: {epoch_dev_loss:.4f} | Acc: {epoch_dev_acc:.2f}%")

## 7. Final evaluation on the unseen Test set

In [ ]:
model.eval()
test_corrects, total_test = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        test_corrects += torch.sum(preds == labels.data)
        total_test += inputs.size(0)

final_test_acc = (test_corrects.double() / total_test) * 100

print("FINAL METRICS")
print(f"  Train accuracy: {epoch_train_acc:.2f}%")
print(f"  Dev accuracy:   {epoch_dev_acc:.2f}%")
print(f"  Test accuracy:  {final_test_acc:.2f}%  ({int(test_corrects)}/{total_test} images)")